In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
df = pd.read_csv("alz-speech.csv")

X = df[["sex", "age", "transcript"]]
y = df["ad"]

text_features = "transcript"
categorical_features = ["sex"]
numeric_features = ["age"]

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95
        ), text_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ]
)

logreg_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall"
}

results = cross_validate(
    logreg_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Logistic Regression Results")
for metric in scoring.keys():
    scores = results[f"test_{metric}"]
    print(f"{metric:10s}: {scores.mean():.4f} +- {scores.std():.4f}")

Logistic Regression Results
accuracy  : 0.7710 +- 0.0802
f1        : 0.7435 +- 0.1026
roc_auc   : 0.8639 +- 0.0657
precision : 0.8340 +- 0.1057
recall    : 0.6830 +- 0.1300


In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC

df = pd.read_csv("alz-speech.csv")

X = df[["sex", "age", "transcript"]]
y = df["ad"]

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95
        ), "transcript"),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), ["sex"]),
    ]
)
svm_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", SVC(
        kernel="linear",
        C=1.0,
        probability=True,
        random_state=42
    ))
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "precision": "precision",
    "recall": "recall"
}

results = cross_validate(
    svm_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Linear SVM Results")
for metric in scoring.keys():
    scores = results[f"test_{metric}"]
    print(f"{metric:10s}: {scores.mean():.4f} +- {scores.std():.4f}")

Linear SVM Results
accuracy  : 0.8010 +- 0.0686
f1        : 0.7726 +- 0.0931
roc_auc   : 0.8766 +- 0.0580
precision : 0.8818 +- 0.0851
recall    : 0.6996 +- 0.1265
